# 00 EDA 01：日级大幅波动归因与特征发现

目标：解释日级别大幅波动，例如月末翘尾、月初骤降、单日峰值占比异常，并把可解释信号沉淀为“每日预测当月总量”的候选特征。

本 notebook 只读取现有 CSV，不保存图片、不写中间文件。所有图表直接在 notebook 内联显示。

分析口径：
- 日粒度实体：`bizym + transdate`
- 目标解释变量：`qty`、`day_share = qty / actual_month_total`、日历基准残差
- 预测任务关注：给定某日已发生的 MTD 信息，预测当月最终总量
- 泄漏原则：`actual_month_total` 只用于 EDA 标签、占比和残差，不作为上线时点可用特征

## 0. 环境、路径与数据合同

优先读取同目录 `Essentiale-daily.csv`。如果从 repo root 运行，也会自动定位到 `code/30d-jenny/Essentiale-daily.csv`。训练锚点文件只用于对齐预测场景，不改变日级 EDA 数据。

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

sns.set_theme(
    style="whitegrid",
    context="notebook",
    rc={
        "grid.color": "#e6e6e6",
        "grid.linewidth": 0.7,
        "grid.alpha": 0.55,
        "axes.edgecolor": "#d0d0d0",
        "axes.linewidth": 0.8,
    },
)

mpl.rcParams.update({
    "figure.dpi": 140,
    "axes.unicode_minus": False,
})

preferred_fonts = [
    "PingFang SC", "Hiragino Sans GB", "Microsoft YaHei", "Heiti SC",
    "Arial Unicode MS", "SimHei", "Noto Sans CJK SC", "DejaVu Sans"
]
for font in preferred_fonts:
    plt.rcParams["font.sans-serif"] = [font] + plt.rcParams.get("font.sans-serif", [])

NOTEBOOK_DIR_CANDIDATES = [Path.cwd(), Path.cwd() / "code/30d-jenny", Path.cwd().parent, Path.cwd().parent.parent]
DATA_CANDIDATES = [
    Path("Essentiale-daily.csv"),
    Path("code/30d-jenny/Essentiale-daily.csv"),
]
DAILY_PATH = None
for base in NOTEBOOK_DIR_CANDIDATES:
    for rel in DATA_CANDIDATES:
        p = base / rel
        if p.exists():
            DAILY_PATH = p.resolve()
            break
    if DAILY_PATH is not None:
        break
if DAILY_PATH is None:
    raise FileNotFoundError("Cannot locate Essentiale-daily.csv from notebook dir or repo root")

DATA_DIR = DAILY_PATH.parent
ANCHOR_FILES = [DATA_DIR / "train.csv", DATA_DIR / "valid.csv", DATA_DIR / "test.csv"]

try:
    from chinese_calendar import is_workday as cn_is_workday
    WORKDAY_SOURCE = "chinese_calendar"
    def is_business_workday(ts):
        return bool(cn_is_workday(pd.Timestamp(ts).date()))
except Exception:
    WORKDAY_SOURCE = "weekday_fallback_Mon_Fri"
    def is_business_workday(ts):
        return pd.Timestamp(ts).weekday() < 5

MONTH_ORDER = list(range(1, 13))
YEAR_COLORS = {
    2022: "#4c72b0", 2023: "#dd8452", 2024: "#55a868",
    2025: "#c44e52", 2026: "#8172b3",
}
PHASE_ORDER = ["first_3_wd", "first_5_wd", "middle_wd", "last_5_wd", "last_3_wd", "non_workday"]


def fmt_num(x):
    if pd.isna(x):
        return ""
    return f"{x:,.0f}"


def fmt_pct(x):
    if pd.isna(x):
        return ""
    return f"{x:.1%}"


def fmt_float(x):
    if pd.isna(x):
        return ""
    return f"{x:,.3f}"


def display_table(df, title=None, n=None):
    if title:
        print(title)
    display(df.head(n) if n else df)


def safe_corr(x, y):
    tmp = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(tmp) < 5 or tmp["x"].nunique() < 2 or tmp["y"].nunique() < 2:
        return np.nan
    return tmp["x"].corr(tmp["y"], method="pearson")

print(f"Daily data: {DAILY_PATH}")
print(f"Workday source: {WORKDAY_SOURCE}")

In [ ]:
df_raw = pd.read_csv(DAILY_PATH)
df = df_raw.copy()
df["transdate"] = pd.to_datetime(df["transdate"])
df["bizym"] = df["bizym"].astype(int)
df = df.sort_values(["transdate", "bizym"]).reset_index(drop=True)

required_cols = {"bizym", "transdate", "num_hosp", "qty"}
missing_cols = sorted(required_cols - set(df.columns))
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# 去掉实际观测重复；若同日多行，则汇总为日粒度。
df = (
    df.groupby(["bizym", "transdate"], as_index=False)
    .agg(num_hosp=("num_hosp", "sum"), qty=("qty", "sum"))
    .sort_values("transdate")
    .reset_index(drop=True)
)

df["year"] = df["bizym"] // 100
df["month"] = df["bizym"] % 100
df["month_start"] = df["transdate"].dt.to_period("M").dt.to_timestamp()
df["date"] = df["transdate"].dt.date
df["day_of_month"] = df["transdate"].dt.day
df["weekday"] = df["transdate"].dt.dayofweek + 1
df["weekday_name"] = df["transdate"].dt.day_name().str.slice(0, 3)
df["days_in_month"] = df["transdate"].dt.days_in_month
df["days_to_month_end"] = df["days_in_month"] - df["day_of_month"]
df["calendar_day_frac"] = df["day_of_month"] / df["days_in_month"]
df["is_workday"] = df["transdate"].map(is_business_workday)
df["is_non_workday"] = ~df["is_workday"]

# 月内星期位置：W1-1 表示该自然月日历网格中的第 1 周周一。
month_week_monday = df["month_start"] - pd.to_timedelta(df["month_start"].dt.dayofweek, unit="D")
df["aligned_week"] = (((df["transdate"] - month_week_monday).dt.days // 7) + 1).astype(int)
df["week_weekday_pos"] = (df["aligned_week"] - 1) * 7 + df["weekday"]
df["week_weekday_label"] = "W" + df["aligned_week"].astype(str) + "-" + df["weekday"].astype(str)

# 工作日序号与倒数工作日序号。
df["workday_seq"] = df.groupby("bizym")["is_workday"].cumsum().where(df["is_workday"], np.nan)
month_workdays = df.groupby("bizym", as_index=False).agg(max_workday_seq=("workday_seq", "max"))
df = df.merge(month_workdays, on="bizym", how="left")
df["workdays_to_month_end"] = df["max_workday_seq"] - df["workday_seq"]
df["workday_frac"] = df["workday_seq"] / df["max_workday_seq"]

# 月度标签与占比标签。
monthly = (
    df.groupby(["bizym", "year", "month", "month_start"], as_index=False)
    .agg(
        actual_month_total=("qty", "sum"),
        month_num_hosp=("num_hosp", "sum"),
        observed_days=("transdate", "nunique"),
        observed_workdays=("is_workday", "sum"),
        max_workday_seq=("max_workday_seq", "max"),
    )
)
monthly["month_label"] = monthly["month_start"].dt.strftime("%Y-%m")
monthly["avg_qty_per_workday"] = monthly["actual_month_total"] / monthly["max_workday_seq"]
monthly["prev_month_total"] = monthly["actual_month_total"].shift(1)
monthly["mom_pct"] = monthly["actual_month_total"] / monthly["prev_month_total"] - 1
monthly["prev_year_bizym"] = (monthly["year"] - 1) * 100 + monthly["month"]
monthly = monthly.merge(
    monthly[["bizym", "actual_month_total"]].rename(columns={"bizym": "prev_year_bizym", "actual_month_total": "prev_year_month_total"}),
    on="prev_year_bizym",
    how="left",
)
monthly["yoy_pct"] = monthly["actual_month_total"] / monthly["prev_year_month_total"] - 1

df = df.merge(monthly[["bizym", "month_label", "actual_month_total", "month_num_hosp", "avg_qty_per_workday"]], on="bizym", how="left")
df["day_share"] = df["qty"] / df["actual_month_total"]
df["hosp_share"] = df["num_hosp"] / df["month_num_hosp"]
df["qty_per_hosp"] = df["qty"] / df["num_hosp"].replace(0, np.nan)
df["log_qty"] = np.log1p(df["qty"])
df["log_num_hosp"] = np.log1p(df["num_hosp"])
df["mtd_qty"] = df.groupby("bizym")["qty"].cumsum()
df["mtd_num_hosp"] = df.groupby("bizym")["num_hosp"].cumsum()
df["mtd_share"] = df["mtd_qty"] / df["actual_month_total"]
df["remaining_qty"] = df["actual_month_total"] - df["mtd_qty"]
df["remaining_share"] = 1 - df["mtd_share"]

# 预测可用的局部动态特征：只依赖历史到当天。
df["prev_day_qty"] = df.groupby("bizym")["qty"].shift(1)
df["prev_workday_qty"] = df.loc[df["is_workday"]].groupby("bizym")["qty"].shift(1).reindex(df.index)
for w in [3, 5, 7]:
    df[f"qty_roll_{w}d"] = df.groupby("bizym")["qty"].transform(lambda s: s.shift(1).rolling(w, min_periods=2).mean())
    df[f"num_hosp_roll_{w}d"] = df.groupby("bizym")["num_hosp"].transform(lambda s: s.shift(1).rolling(w, min_periods=2).mean())
df["daily_jump_vs_prev"] = df["qty"] / df["prev_day_qty"] - 1
df["daily_jump_vs_roll_5d"] = df["qty"] / df["qty_roll_5d"] - 1

# 月内阶段：便于把“月初骤降 / 月末翘尾”定量化。
def assign_phase(row):
    if not row["is_workday"] or pd.isna(row["workday_seq"]):
        return "non_workday"
    if row["workday_seq"] <= 3:
        return "first_3_wd"
    if row["workday_seq"] <= 5:
        return "first_5_wd"
    if row["workdays_to_month_end"] <= 2:
        return "last_3_wd"
    if row["workdays_to_month_end"] <= 4:
        return "last_5_wd"
    return "middle_wd"

df["month_phase"] = df.apply(assign_phase, axis=1)

summary = pd.DataFrame({
    "metric": ["rows", "months", "date_min", "date_max", "qty_sum", "num_hosp_sum", "workday_source"],
    "value": [
        len(df), df["bizym"].nunique(), df["transdate"].min().date(), df["transdate"].max().date(),
        fmt_num(df["qty"].sum()), fmt_num(df["num_hosp"].sum()), WORKDAY_SOURCE,
    ],
})

display_table(summary, "Data contract / snapshot")
display_table(monthly.tail(12), "Recent monthly totals")

## 1. 全局走势：先定位“异常”的背景

先看月总量、日销量和医院数的长期背景。日级异常如果同时伴随 `num_hosp` 跳变，通常更像覆盖医院数量变化；如果 `qty_per_hosp` 同时异常，可能是单院放量、补货或月底集中开票等行为。

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=False)

axes[0].plot(monthly["month_start"], monthly["actual_month_total"], marker="o", linewidth=2, color="#2f6f73")
axes[0].set_title("Monthly actual total")
axes[0].set_ylabel("qty")
axes[0].yaxis.set_major_formatter(mpl.ticker.StrMethodFormatter("{x:,.0f}"))

axes[1].plot(df["transdate"], df["qty"], color="#4c72b0", linewidth=0.8, alpha=0.75, label="daily qty")
axes[1].plot(df["transdate"], df["qty"].rolling(14, min_periods=4).mean(), color="#c44e52", linewidth=1.8, label="14d MA")
axes[1].set_title("Daily qty with 14-day moving average")
axes[1].set_ylabel("qty")
axes[1].legend()
axes[1].yaxis.set_major_formatter(mpl.ticker.StrMethodFormatter("{x:,.0f}"))

axes[2].plot(df["transdate"], df["num_hosp"], color="#8172b3", linewidth=0.8, alpha=0.7, label="daily num_hosp")
axes[2].plot(df["transdate"], df["num_hosp"].rolling(14, min_periods=4).mean(), color="#dd8452", linewidth=1.8, label="14d MA")
axes[2].set_title("Daily num_hosp with 14-day moving average")
axes[2].set_ylabel("num_hosp")
axes[2].legend()
axes[2].yaxis.set_major_formatter(mpl.ticker.StrMethodFormatter("{x:,.0f}"))

for ax in axes:
    ax.grid(True, alpha=0.35)
plt.tight_layout()

## 2. 月初骤降与月末翘尾：阶段贡献表

用工作日阶段而不是自然日阶段观察：
- `first_3_wd`：前 3 个工作日
- `first_5_wd`：第 4-5 个工作日
- `middle_wd`：中段工作日
- `last_5_wd`：倒数第 5-3 个工作日
- `last_3_wd`：最后 3 个工作日

核心看 `avg_day_share` 和相对中段的 `uplift_vs_middle`。如果月末阶段持续大于 1，说明月末翘尾是稳定结构，不只是偶发峰值。

In [ ]:
phase_daily = df[df["is_workday"]].copy()
phase_summary = (
    phase_daily.groupby(["year", "month", "bizym", "month_phase"], as_index=False)
    .agg(
        days=("transdate", "nunique"),
        phase_qty=("qty", "sum"),
        avg_day_qty=("qty", "mean"),
        avg_day_share=("day_share", "mean"),
        avg_qty_per_hosp=("qty_per_hosp", "mean"),
    )
)
phase_mid = phase_summary.query("month_phase == 'middle_wd'")[["bizym", "avg_day_share", "avg_day_qty"]].rename(
    columns={"avg_day_share": "middle_avg_day_share", "avg_day_qty": "middle_avg_day_qty"}
)
phase_summary = phase_summary.merge(phase_mid, on="bizym", how="left")
phase_summary["uplift_vs_middle"] = phase_summary["avg_day_share"] / phase_summary["middle_avg_day_share"]
phase_summary["qty_uplift_vs_middle"] = phase_summary["avg_day_qty"] / phase_summary["middle_avg_day_qty"]

phase_overall = (
    phase_summary.groupby(["month", "month_phase"], as_index=False)
    .agg(
        n_months=("bizym", "nunique"),
        avg_day_share=("avg_day_share", "mean"),
        median_day_share=("avg_day_share", "median"),
        uplift_vs_middle=("uplift_vs_middle", "mean"),
        qty_uplift_vs_middle=("qty_uplift_vs_middle", "mean"),
        avg_qty_per_hosp=("avg_qty_per_hosp", "mean"),
    )
)
phase_overall["month_phase"] = pd.Categorical(phase_overall["month_phase"], PHASE_ORDER, ordered=True)
phase_overall = phase_overall.sort_values(["month", "month_phase"])

phase_display = phase_overall.copy()
for c in ["avg_day_share", "median_day_share"]:
    phase_display[c] = phase_display[c].map(fmt_pct)
for c in ["uplift_vs_middle", "qty_uplift_vs_middle", "avg_qty_per_hosp"]:
    phase_display[c] = phase_display[c].map(fmt_float)
display_table(phase_display, "Month-phase contribution summary")

pivot_uplift = phase_overall.pivot(index="month", columns="month_phase", values="uplift_vs_middle").reindex(columns=PHASE_ORDER)
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(pivot_uplift, annot=True, fmt=".2f", cmap="RdBu_r", center=1.0, linewidths=0.5, ax=ax)
ax.set_title("Average daily-share uplift vs middle workdays by calendar month")
ax.set_xlabel("month phase")
ax.set_ylabel("calendar month")
plt.tight_layout()

## 3. 峰值比例关系异常：单日峰值、Top3 贡献与峰值/中位数比

预测月总量时，单日峰值很容易把 MTD 斜率拉偏。这里用三类指标识别：
- `max_day_share`：最大单日贡献占当月比例
- `top3_day_share`：前三大日贡献占当月比例
- `peak_to_workday_median`：最大工作日销量 / 当月工作日中位数

In [ ]:
workday_df = df[df["is_workday"]].copy()
peak_rows = []
for bizym, g in workday_df.groupby("bizym"):
    g = g.sort_values("qty", ascending=False).copy()
    top = g.iloc[0]
    peak_rows.append({
        "bizym": bizym,
        "year": int(top["year"]),
        "month": int(top["month"]),
        "peak_date": top["transdate"].date(),
        "peak_week_weekday": top["week_weekday_label"],
        "peak_workday_seq": int(top["workday_seq"]),
        "peak_workdays_to_end": int(top["workdays_to_month_end"]),
        "peak_phase": top["month_phase"],
        "peak_qty": top["qty"],
        "peak_num_hosp": top["num_hosp"],
        "peak_day_share": top["day_share"],
        "top3_day_share": g.head(3)["day_share"].sum(),
        "workday_median_qty": g["qty"].median(),
        "peak_to_workday_median": top["qty"] / g["qty"].median() if g["qty"].median() else np.nan,
        "peak_qty_per_hosp": top["qty_per_hosp"],
    })
peak_table = pd.DataFrame(peak_rows).merge(monthly[["bizym", "actual_month_total", "yoy_pct", "mom_pct"]], on="bizym", how="left")
peak_table = peak_table.sort_values("peak_day_share", ascending=False)

peak_display = peak_table.head(20).copy()
for c in ["peak_qty", "peak_num_hosp", "workday_median_qty", "actual_month_total"]:
    peak_display[c] = peak_display[c].map(fmt_num)
for c in ["peak_day_share", "top3_day_share", "yoy_pct", "mom_pct"]:
    peak_display[c] = peak_display[c].map(fmt_pct)
for c in ["peak_to_workday_median", "peak_qty_per_hosp"]:
    peak_display[c] = peak_display[c].map(fmt_float)
display_table(peak_display, "Top 20 peak-share months")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.scatterplot(
    data=peak_table,
    x="peak_day_share",
    y="top3_day_share",
    hue="year",
    size="peak_to_workday_median",
    palette=YEAR_COLORS,
    sizes=(40, 220),
    ax=axes[0],
)
axes[0].xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1))
axes[0].yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1))
axes[0].set_title("Peak day share vs top-3 day share")
axes[0].set_xlabel("max single workday share")
axes[0].set_ylabel("top 3 workdays share")

sns.boxplot(data=peak_table, x="month", y="peak_to_workday_median", color="#9ecae1", ax=axes[1])
sns.stripplot(data=peak_table, x="month", y="peak_to_workday_median", hue="year", palette=YEAR_COLORS, dodge=False, ax=axes[1])
axes[1].set_title("Peak / monthly workday median by calendar month")
axes[1].set_xlabel("calendar month")
axes[1].set_ylabel("peak_to_workday_median")
axes[1].legend(title="year", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

## 4. 日历位置基准残差：找出“不是正常月内节奏”的异常日

先建立一个只由 `month + W周-星期 + 工作日/非工作日` 决定的基准期望：同一自然月、同一日历格子位置的历史中位数。

残差定义：
`profile_resid = log1p(qty) - median(log1p(qty) | month, week_weekday_label, is_workday)`

这样可把“每个月本来就月底高”与“该日相对同位置异常高/低”分开。

In [ ]:
profile_keys = ["month", "week_weekday_label", "is_workday"]
profile = (
    df.groupby(profile_keys, as_index=False)
    .agg(
        profile_n=("qty", "size"),
        expected_log_qty=("log_qty", "median"),
        expected_day_share=("day_share", "median"),
        expected_num_hosp=("num_hosp", "median"),
    )
)
df_resid = df.merge(profile, on=profile_keys, how="left")
df_resid["profile_resid"] = df_resid["log_qty"] - df_resid["expected_log_qty"]
df_resid["share_resid"] = df_resid["day_share"] - df_resid["expected_day_share"]
df_resid["num_hosp_resid_pct"] = df_resid["num_hosp"] / df_resid["expected_num_hosp"].replace(0, np.nan) - 1

top_pos = df_resid.sort_values("profile_resid", ascending=False).head(20).copy()
top_neg = df_resid.sort_values("profile_resid", ascending=True).head(20).copy()
resid_rank = pd.concat([top_pos.assign(direction="positive_spike"), top_neg.assign(direction="negative_drop")], ignore_index=True)
resid_rank = resid_rank[[
    "direction", "transdate", "bizym", "year", "month", "weekday_name", "week_weekday_label",
    "workday_seq", "workdays_to_month_end", "month_phase", "qty", "expected_log_qty",
    "profile_resid", "day_share", "share_resid", "num_hosp", "num_hosp_resid_pct", "qty_per_hosp"
]]
resid_display = resid_rank.copy()
for c in ["qty", "num_hosp"]:
    resid_display[c] = resid_display[c].map(fmt_num)
for c in ["day_share", "share_resid", "num_hosp_resid_pct"]:
    resid_display[c] = resid_display[c].map(fmt_pct)
for c in ["expected_log_qty", "profile_resid", "qty_per_hosp"]:
    resid_display[c] = resid_display[c].map(fmt_float)
display_table(resid_display, "Largest positive and negative residual days vs calendar-position profile")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(df_resid["profile_resid"].dropna(), bins=45, kde=True, color="#4c72b0", ax=axes[0])
axes[0].axvline(df_resid["profile_resid"].quantile(0.95), color="#c44e52", linestyle="--", label="p95")
axes[0].axvline(df_resid["profile_resid"].quantile(0.05), color="#c44e52", linestyle="--", label="p05")
axes[0].set_title("Distribution of profile residual")
axes[0].legend()

resid_month_phase = df_resid[df_resid["is_workday"]].groupby(["month", "month_phase"], as_index=False).agg(mean_resid=("profile_resid", "mean"))
resid_month_phase["month_phase"] = pd.Categorical(resid_month_phase["month_phase"], PHASE_ORDER, ordered=True)
heat = resid_month_phase.pivot(index="month", columns="month_phase", values="mean_resid").reindex(columns=PHASE_ORDER)
sns.heatmap(heat, annot=True, fmt=".2f", cmap="RdBu_r", center=0, linewidths=0.5, ax=axes[1])
axes[1].set_title("Mean residual by month and phase")
plt.tight_layout()

## 5. Pearson IC：哪些变量解释日级波动

这里把 IC 当作“变量与日级目标的 Pearson 相关系数”。分三类目标：
- `day_share`：解释当日对月总量的贡献
- `profile_resid`：解释剔除日历位置基准后的异常
- `remaining_share`：贴近“当天之后还剩多少月总量”的预测目标

注意：某些变量只用于 EDA 标签或训练监督，不一定可在预测时点作为特征。

In [ ]:
analysis = df_resid.copy()
analysis["is_first_3_wd"] = (analysis["month_phase"] == "first_3_wd").astype(int)
analysis["is_first_5_wd"] = analysis["month_phase"].isin(["first_3_wd", "first_5_wd"]).astype(int)
analysis["is_last_5_wd"] = analysis["month_phase"].isin(["last_5_wd", "last_3_wd"]).astype(int)
analysis["is_last_3_wd"] = (analysis["month_phase"] == "last_3_wd").astype(int)
analysis["is_monday"] = (analysis["weekday"] == 1).astype(int)
analysis["is_friday"] = (analysis["weekday"] == 5).astype(int)
analysis["log_qty_per_hosp"] = np.log1p(analysis["qty_per_hosp"])
analysis["num_hosp_vs_profile"] = analysis["num_hosp"] / analysis["expected_num_hosp"].replace(0, np.nan)
analysis["mtd_velocity_vs_linear"] = analysis["mtd_share"] - analysis["calendar_day_frac"]
analysis["mtd_velocity_vs_workday"] = analysis["mtd_share"] - analysis["workday_frac"]
analysis["prev_day_log_qty"] = np.log1p(analysis["prev_day_qty"])
analysis["roll_5d_log_qty"] = np.log1p(analysis["qty_roll_5d"])

feature_cols = [
    "num_hosp", "log_num_hosp", "qty_per_hosp", "log_qty_per_hosp", "num_hosp_vs_profile",
    "day_of_month", "days_to_month_end", "weekday", "aligned_week", "week_weekday_pos",
    "workday_seq", "workdays_to_month_end", "workday_frac", "calendar_day_frac",
    "is_workday", "is_non_workday", "is_monday", "is_friday", "is_first_3_wd", "is_first_5_wd", "is_last_5_wd", "is_last_3_wd",
    "mtd_qty", "mtd_num_hosp", "mtd_share", "remaining_share", "mtd_velocity_vs_linear", "mtd_velocity_vs_workday",
    "prev_day_log_qty", "roll_5d_log_qty", "daily_jump_vs_prev", "daily_jump_vs_roll_5d",
]
target_cols = ["day_share", "profile_resid", "remaining_share"]

ic_rows = []
for target in target_cols:
    for feat in feature_cols:
        if feat not in analysis.columns:
            continue
        x = analysis[feat].astype(float) if analysis[feat].dtype == bool else analysis[feat]
        ic_rows.append({
            "target": target,
            "feature": feat,
            "pearson_ic": safe_corr(x, analysis[target]),
            "n": analysis[[feat, target]].replace([np.inf, -np.inf], np.nan).dropna().shape[0],
        })
ic = pd.DataFrame(ic_rows)
ic["abs_ic"] = ic["pearson_ic"].abs()
ic = ic.sort_values(["target", "abs_ic"], ascending=[True, False])

ic_display = ic.groupby("target", group_keys=False).head(15).copy()
ic_display["pearson_ic"] = ic_display["pearson_ic"].map(fmt_float)
ic_display["abs_ic"] = ic_display["abs_ic"].map(fmt_float)
display_table(ic_display, "Top Pearson IC by target")

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)
for ax, target in zip(axes, target_cols):
    plot_ic = ic.query("target == @target").head(12).sort_values("pearson_ic")
    colors = np.where(plot_ic["pearson_ic"] >= 0, "#55a868", "#c44e52")
    ax.barh(plot_ic["feature"], plot_ic["pearson_ic"], color=colors)
    ax.axvline(0, color="0.25", linewidth=1)
    ax.set_title(f"Pearson IC -> {target}")
    ax.set_xlabel("Pearson r")
plt.tight_layout()

## 6. 残差模型：变量组合能解释多少异常波动

用一个轻量线性残差模型衡量可解释度：

`log_qty ~ log_num_hosp + calendar month + weekday + month phase + week-weekday position`

解释逻辑：
- 如果加入 `log_num_hosp` 后残差显著收敛，说明很多峰值来自覆盖医院数变化。
- 如果 `month_phase` 或 `week_weekday_label` 的系数明显，说明是可被日历位置编码吸收的节奏。
- 模型只是 EDA 解释器，不作为最终预测模型。

In [ ]:
model_df = df_resid.copy()
model_df = model_df.dropna(subset=["log_qty", "log_num_hosp", "month_phase", "week_weekday_label"])

base_cols = ["log_num_hosp"]
cat_cols = ["month", "weekday", "month_phase", "week_weekday_label"]
X = model_df[base_cols].copy()
for c in cat_cols:
    dummies = pd.get_dummies(model_df[c].astype(str), prefix=c, drop_first=True, dtype=float)
    X = pd.concat([X, dummies], axis=1)
X.insert(0, "intercept", 1.0)
y = model_df["log_qty"].astype(float).to_numpy()
X_mat = X.astype(float).to_numpy()
coef, *_ = np.linalg.lstsq(X_mat, y, rcond=None)
model_df["pred_log_qty"] = X_mat @ coef
model_df["model_resid"] = model_df["log_qty"] - model_df["pred_log_qty"]

sst = float(((y - y.mean()) ** 2).sum())
sse = float((model_df["model_resid"] ** 2).sum())
r2 = 1 - sse / sst

coef_table = pd.DataFrame({"term": X.columns, "coef": coef})
coef_table["abs_coef"] = coef_table["coef"].abs()
coef_table = coef_table.sort_values("abs_coef", ascending=False)

print(f"EDA residual model R^2: {r2:.3f}")
display_table(coef_table.head(30), "Largest absolute coefficients")

# 组合解释表：月内阶段 x 星期 x 医院覆盖强弱。
model_df["num_hosp_bucket"] = pd.qcut(model_df["num_hosp_vs_profile"].replace([np.inf, -np.inf], np.nan), q=4, labels=["low_hosp", "mid_low_hosp", "mid_high_hosp", "high_hosp"], duplicates="drop")
combo = (
    model_df.groupby(["month_phase", "weekday_name", "num_hosp_bucket"], observed=False, as_index=False)
    .agg(
        n=("qty", "size"),
        avg_qty=("qty", "mean"),
        avg_day_share=("day_share", "mean"),
        avg_profile_resid=("profile_resid", "mean"),
        avg_model_resid=("model_resid", "mean"),
        avg_num_hosp_vs_profile=("num_hosp_vs_profile", "mean"),
    )
    .query("n >= 6")
    .sort_values("avg_profile_resid", ascending=False)
)
combo_display = pd.concat([combo.head(15).assign(direction="high_resid_combo"), combo.tail(15).assign(direction="low_resid_combo")], ignore_index=True)
for c in ["avg_day_share"]:
    combo_display[c] = combo_display[c].map(fmt_pct)
for c in ["avg_qty"]:
    combo_display[c] = combo_display[c].map(fmt_num)
for c in ["avg_profile_resid", "avg_model_resid", "avg_num_hosp_vs_profile"]:
    combo_display[c] = combo_display[c].map(fmt_float)
display_table(combo_display, "Variable combinations explaining high/low residual days")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.scatterplot(data=model_df, x="profile_resid", y="model_resid", hue="month_phase", alpha=0.65, ax=axes[0])
axes[0].axhline(0, color="0.25", linewidth=1)
axes[0].axvline(0, color="0.25", linewidth=1)
axes[0].set_title("Calendar-profile residual vs richer residual model")
axes[0].legend(bbox_to_anchor=(1.02, 1), loc="upper left")

resid_compare = pd.DataFrame({
    "residual_type": ["profile_resid", "model_resid"],
    "std": [model_df["profile_resid"].std(), model_df["model_resid"].std()],
    "p95_abs": [model_df["profile_resid"].abs().quantile(0.95), model_df["model_resid"].abs().quantile(0.95)],
})
sns.barplot(data=resid_compare.melt(id_vars="residual_type"), x="variable", y="value", hue="residual_type", ax=axes[1])
axes[1].set_title("Residual compression after variable combination")
axes[1].set_xlabel("metric")
plt.tight_layout()

display_table(resid_compare, "Residual compression metrics")

## 7. 预测锚点视角：D-10 / D-5 时点还能解释多少月总量

每日预测当月总量时，最重要的是：
- 截至当天 MTD 已完成比例是否稳定
- 后续剩余比例是否受月末阶段、当月日历形态、医院覆盖强弱影响
- 当前 MTD 速度相对“线性进度/工作日进度”是提前还是落后

In [ ]:
anchor_offsets = [10, 5]
anchor_daily = df_resid[df_resid["is_workday"] & df_resid["workdays_to_month_end"].isin([o - 1 for o in anchor_offsets])].copy()
anchor_daily["forecast_offset"] = anchor_daily["workdays_to_month_end"].astype(int) + 1
anchor_daily["anchor_mtd_qty_ratio_to_month"] = anchor_daily["mtd_share"]
anchor_daily["remaining_qty_ratio_to_month"] = anchor_daily["remaining_share"]
anchor_daily["mtd_qty_per_elapsed_workday"] = anchor_daily["mtd_qty"] / anchor_daily["workday_seq"]
anchor_daily["needed_avg_remaining_workday_qty"] = anchor_daily["remaining_qty"] / anchor_daily["forecast_offset"]
anchor_daily["remaining_vs_elapsed_rate"] = anchor_daily["needed_avg_remaining_workday_qty"] / anchor_daily["mtd_qty_per_elapsed_workday"]
anchor_daily["mtd_vs_workday_linear"] = anchor_daily["mtd_share"] - anchor_daily["workday_frac"]

anchor_summary = (
    anchor_daily.groupby(["month", "forecast_offset"], as_index=False)
    .agg(
        n=("bizym", "nunique"),
        avg_mtd_share=("mtd_share", "mean"),
        std_mtd_share=("mtd_share", "std"),
        avg_remaining_share=("remaining_share", "mean"),
        avg_remaining_vs_elapsed_rate=("remaining_vs_elapsed_rate", "mean"),
        std_remaining_vs_elapsed_rate=("remaining_vs_elapsed_rate", "std"),
    )
)
anchor_display = anchor_summary.copy()
for c in ["avg_mtd_share", "std_mtd_share", "avg_remaining_share"]:
    anchor_display[c] = anchor_display[c].map(fmt_pct)
for c in ["avg_remaining_vs_elapsed_rate", "std_remaining_vs_elapsed_rate"]:
    anchor_display[c] = anchor_display[c].map(fmt_float)
display_table(anchor_display, "D-10 / D-5 anchor stability by month")

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, offset in zip(axes, anchor_offsets):
    g = anchor_daily.query("forecast_offset == @offset")
    sns.lineplot(data=g, x="month", y="mtd_share", hue="year", palette=YEAR_COLORS, marker="o", ax=ax)
    ax.set_title(f"MTD share at D-{offset} workday anchor")
    ax.set_xlabel("calendar month")
    ax.set_ylabel("MTD share")
    ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1))
    ax.set_xticks(MONTH_ORDER)
plt.tight_layout()

anchor_feature_cols = [
    "forecast_offset", "month", "max_workday_seq", "workday_seq", "workday_frac",
    "mtd_qty", "mtd_num_hosp", "mtd_share", "mtd_vs_workday_linear",
    "num_hosp", "log_num_hosp", "qty", "day_share", "profile_resid", "remaining_vs_elapsed_rate",
]
anchor_ic_rows = []
for feat in anchor_feature_cols:
    anchor_ic_rows.append({
        "feature": feat,
        "pearson_ic_to_actual_month_total": safe_corr(anchor_daily[feat], anchor_daily["actual_month_total"]),
        "pearson_ic_to_remaining_share": safe_corr(anchor_daily[feat], anchor_daily["remaining_share"]),
        "n": anchor_daily[[feat, "actual_month_total", "remaining_share"]].dropna().shape[0],
    })
anchor_ic = pd.DataFrame(anchor_ic_rows)
anchor_ic["abs_ic_month_total"] = anchor_ic["pearson_ic_to_actual_month_total"].abs()
anchor_ic = anchor_ic.sort_values("abs_ic_month_total", ascending=False)
anchor_ic_display = anchor_ic.copy()
for c in ["pearson_ic_to_actual_month_total", "pearson_ic_to_remaining_share", "abs_ic_month_total"]:
    anchor_ic_display[c] = anchor_ic_display[c].map(fmt_float)
display_table(anchor_ic_display, "Anchor-level Pearson IC")

## 8. 可进入 ML 的候选特征与异常规律总结

下面的表不是最终模型字段，而是“优先试验清单”。其中 `leakage_risk = high` 的字段只用于训练标签、误差分析或构造历史统计，不能在预测日直接用当月最终值。

In [ ]:
feature_plan = pd.DataFrame([
    {
        "feature_family": "月内工作日位置",
        "candidate_features": "workday_seq, workdays_to_month_end, workday_frac, forecast_offset",
        "explains": "月初骤降、月末翘尾、D-10/D-5 剩余比例",
        "availability_at_forecast": "yes",
        "leakage_risk": "low",
    },
    {
        "feature_family": "日历格子位置",
        "candidate_features": "weekday, aligned_week, week_weekday_label, is_monday/is_friday",
        "explains": "同月 W周-星期 对齐后的稳定峰谷",
        "availability_at_forecast": "yes",
        "leakage_risk": "low",
    },
    {
        "feature_family": "月内阶段分桶",
        "candidate_features": "first_3_wd, first_5_wd, middle_wd, last_5_wd, last_3_wd",
        "explains": "月初骤降 / 月末翘尾的可解释结构",
        "availability_at_forecast": "yes",
        "leakage_risk": "low",
    },
    {
        "feature_family": "医院覆盖强度",
        "candidate_features": "num_hosp, log_num_hosp, mtd_num_hosp, num_hosp_vs_historical_profile",
        "explains": "峰值是否来自覆盖医院数同步上升",
        "availability_at_forecast": "same-day actual or lagged depends on serving",
        "leakage_risk": "medium",
    },
    {
        "feature_family": "MTD 速度",
        "candidate_features": "mtd_qty, mtd_share_proxy, mtd_vs_workday_linear, rolling_5d_qty",
        "explains": "当前进度相对线性/工作日进度超前或落后",
        "availability_at_forecast": "yes for mtd_qty; mtd_share needs predicted denominator proxy",
        "leakage_risk": "medium/high",
    },
    {
        "feature_family": "历史同位置基准",
        "candidate_features": "median_day_share_by_month_weekday_pos, median_qty_by_month_phase_weekday, profile_resid_lag",
        "explains": "把正常月内节奏和异常残差分开",
        "availability_at_forecast": "yes if computed from past months only",
        "leakage_risk": "low if point-in-time; high if uses current/future month",
    },
    {
        "feature_family": "峰值集中度标签",
        "candidate_features": "max_day_share, top3_day_share, peak_to_workday_median",
        "explains": "解释历史异常月份，不建议直接作为预测日前特征",
        "availability_at_forecast": "partial only after peak occurs",
        "leakage_risk": "high",
    },
])
display_table(feature_plan, "Feature discovery plan")

# 自动生成简短规律摘要，避免只靠肉眼看图。
phase_signal = phase_overall.copy()
last_signal = phase_signal[phase_signal["month_phase"].isin(["last_5_wd", "last_3_wd"])].sort_values("uplift_vs_middle", ascending=False).head(8)
first_signal = phase_signal[phase_signal["month_phase"].isin(["first_3_wd", "first_5_wd"])].sort_values("uplift_vs_middle", ascending=True).head(8)
peak_signal = peak_table.head(8)
resid_signal = combo.sort_values("avg_profile_resid", ascending=False).head(8)

print("Key discovered regularities")
print("1) Strongest month-end tail candidates:")
for _, r in last_signal.iterrows():
    print(f"   - month={int(r['month']):02d}, phase={r['month_phase']}: daily share uplift vs middle = {r['uplift_vs_middle']:.2f}x")
print("2) Strongest month-start drop candidates:")
for _, r in first_signal.iterrows():
    print(f"   - month={int(r['month']):02d}, phase={r['month_phase']}: daily share uplift vs middle = {r['uplift_vs_middle']:.2f}x")
print("3) Peak concentration candidates:")
for _, r in peak_signal.iterrows():
    print(f"   - {r['peak_date']} bizym={int(r['bizym'])}: peak share={r['peak_day_share']:.1%}, top3 share={r['top3_day_share']:.1%}, phase={r['peak_phase']}")
print("4) High-residual variable combinations:")
for _, r in resid_signal.iterrows():
    print(f"   - phase={r['month_phase']}, weekday={r['weekday_name']}, hosp_bucket={r['num_hosp_bucket']}: mean profile residual={r['avg_profile_resid']:.2f}, n={int(r['n'])}")

## 9. 建模建议

推荐下一步实验：
1. 建一个只用预测时点可用字段的 baseline：`anchor_mtd_qty + workday_frac + month + forecast_offset + historical same-month profile`。
2. 增加日历格子统计：过去年份同 `month + week_weekday_label` 的 `median_day_share`、`median_remaining_share_at_anchor`。
3. 增加医院覆盖强度：`mtd_num_hosp`、过去 5 日 `num_hosp` 均值、相对历史同位置医院数 uplift。
4. 对 1/2/3 月和月末阶段单独做 slice eval，因为这些区域的残差和峰值集中度更容易驱动月总量误差。
5. 所有历史 profile 特征必须 point-in-time 计算：预测 2025-03 时只能用 2025-03 之前的数据，不能用完整 2025-03 或未来年份。